# Lab 26 (No Redis) — Optimize ClaimsIQ Using an In-Memory Cache

**Module:** Performance Optimization · Day 15 · Session 01
**Duration:** ~40-50 minutes

### What you will do
Add a cache-aside layer in front of `get_customer_profile`, exactly like
the original Redis version — but backed by a plain Python dictionary
instead of an external Redis server. Same teaching goals: measure the
real latency difference between a cache miss and a cache hit, and test
both TTL expiry and manual invalidation.

### Why no Redis
This version has **zero external dependencies** — no server to install,
no `redis-server` process to keep running. The trade-off, worth naming
explicitly: an in-memory dict only persists for the life of this Python
process. Restart the kernel, and the cache is gone — unlike Redis, which
keeps state independently of any one notebook. That's the real thing
Redis buys you in production, and it's worth saying out loud even though
this lab doesn't need it to teach the caching concept itself.

### Prerequisite
ClaimsIQ Notebooks 00-01 must have run already, so
`mcp_snowflake_server.py` exists on disk.

## Step 1 — Connect (no Redis install needed)

In [1]:
%pip install -q snowflake-connector-python

Note: you may need to restart the kernel to use updated packages.


In [2]:
import time, json
from mcp_snowflake_server import claims_server, SimpleMCPClient
from dotenv import load_dotenv

load_dotenv()

mcp_client = SimpleMCPClient(claims_server)
mcp_client.connect()
print("Connected. No external cache server required for this version.")

Connected to 'claimsiq-snowflake'. Discovered 6 tools.
Connected. No external cache server required for this version.


## Step 2 — Write a minimal in-memory cache class

Same cache-aside SHAPE as the Redis version's `get_customer_profile_cached()`
— `get`, `set` with a TTL, and `delete` for manual invalidation — just
backed by a dict instead of a network call to Redis.

In [3]:
class InMemoryCache:
    """A minimal stand-in for Redis: same interface shape (get/set/delete
    with TTL), backed by a plain dict. Lives only as long as this kernel."""

    def __init__(self):
        self._store = {}  # key -> (value, expires_at)

    def get(self, key: str):
        entry = self._store.get(key)
        if entry is None:
            return None
        value, expires_at = entry
        if expires_at is not None and time.time() > expires_at:
            del self._store[key]  # lazily expire on read, like Redis TTL
            return None
        return value

    def setex(self, key: str, ttl_seconds: int, value):
        expires_at = time.time() + ttl_seconds
        self._store[key] = (value, expires_at)

    def delete(self, key: str):
        self._store.pop(key, None)

cache = InMemoryCache()
print("InMemoryCache ready — same get/setex/delete interface as the Redis client.")

InMemoryCache ready — same get/setex/delete interface as the Redis client.


## Step 3 — Write `get_customer_profile_cached()`

Identical cache-aside logic to the Redis version — only the underlying
`cache` object changed.

In [4]:
CACHE_TTL_SECONDS = 300  # 5 minutes

def get_customer_profile_cached(customer_id: str, verbose=True) -> dict:
    cache_key = f"profile:{customer_id}"
    cached = cache.get(cache_key)
    if cached:
        if verbose:
            print(f"[CACHE HIT] {cache_key}")
        return cached

    if verbose:
        print(f"[CACHE MISS] {cache_key} — calling Snowflake via MCP")
    result = mcp_client.call_tool("get_customer_profile", customer_id=customer_id)
    cache.setex(cache_key, CACHE_TTL_SECONDS, result)
    return result

print("get_customer_profile_cached() ready.")

get_customer_profile_cached() ready.


## Step 4 — Clear any leftover cache, then time an uncached call (cache miss)

In [5]:
cache.delete("profile:CUST99001")  # ensure a clean miss

start = time.time()
result = get_customer_profile_cached("CUST99001")
miss_latency = time.time() - start
print(f"\nCache MISS latency: {miss_latency*1000:.1f}ms")

[CACHE MISS] profile:CUST99001 — calling Snowflake via MCP

Cache MISS latency: 4902.1ms


## Step 5 — Time the SAME call again (cache hit)

In [6]:
start = time.time()
result = get_customer_profile_cached("CUST99001")
hit_latency = time.time() - start
print(f"Cache HIT latency: {hit_latency*1000:.1f}ms")

[CACHE HIT] profile:CUST99001
Cache HIT latency: 0.3ms


## Step 6 — Compare the latency difference

Worth noting explicitly: an in-memory dict lookup is even faster than a
Redis hit would be, since there's no network round trip to a separate
process at all — the speedup here will likely look even more dramatic
than the Redis version's numbers.

In [7]:
speedup = miss_latency / hit_latency if hit_latency > 0 else float("inf")
print(f"Cache miss: {miss_latency*1000:.1f}ms")
print(f"Cache hit:  {hit_latency*1000:.2f}ms")
print(f"Speedup:    {speedup:.1f}x faster on a cache hit")

Cache miss: 4902.1ms
Cache hit:  0.27ms
Speedup:    18244.1x faster on a cache hit


## Step 7 — Set a short TTL, confirm expiry works

In [8]:
cache.delete("profile:CUST99001")
cache.setex("profile:CUST99001", 3, mcp_client.call_tool("get_customer_profile", customer_id="CUST99001"))
print("Cached with a 3-second TTL.")

print("Immediately after caching:", cache.get("profile:CUST99001") is not None)
time.sleep(4)
print("After waiting 4 seconds:  ", cache.get("profile:CUST99001") is not None)

Cached with a 3-second TTL.
Immediately after caching: True
After waiting 4 seconds:   False


## Step 8 — Manually invalidate on a simulated data change

Same event-based invalidation idea as the Redis version: if a new fraud
signal were written for this customer elsewhere in the system, the
STALE cached profile should be cleared immediately, rather than waiting
for the TTL to expire naturally.

In [9]:
# Populate the cache again
get_customer_profile_cached("CUST99001", verbose=False)
print("Cached before 'data change':", cache.get("profile:CUST99001") is not None)

def invalidate_customer_cache(customer_id: str):
    cache.delete(f"profile:{customer_id}")
    print(f"Invalidated cache for {customer_id} due to a data change.")

invalidate_customer_cache("CUST99001")
print("Cached after invalidation:  ", cache.get("profile:CUST99001") is not None)

Cached before 'data change': True
Invalidated cache for CUST99001 due to a data change.
Cached after invalidation:   False


## Deliverable

1. The miss/hit latency numbers from Steps 4-6.
2. Confirmation from Step 7 that the TTL expiry worked as expected.
3. One paragraph: this notebook's `InMemoryCache` and the original Redis
   version expose the exact same `get`/`setex`/`delete` interface. If
   ClaimsIQ later needed to run as multiple separate processes (say, one
   API server per region), would `InMemoryCache` still work correctly?
   Explain specifically why or why not, and what that tells you about
   when the extra Redis setup is actually worth it versus when a plain
   dict is genuinely good enough.